# Importing necessary libraries for SLM Training

In [1]:
import torch
import torch.nn as nn


In [2]:
import os, time, math, random
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
import torch.nn.functional as F

In [3]:
from tqdm import tqdm

# Root Mean Square Normalization

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

# Rotary Positional Embeddings (RoPE)

In [5]:
def rotate_half(x):
    x1 = x[..., :x.shape[-1]//2]
    x2 = x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, rope_sin, rope_cos):
    return (x * rope_cos) + (rotate_half(x) * rope_sin)

# Multi-Head Self Attention (Causal)

In [6]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, mask, rope_sin, rope_cos):
        B, T, C = x.shape

        qkv = self.qkv(x)  # (B, T, 3*C)
        q, k, v = qkv.chunk(3, dim=-1)

        # reshape -> (B, heads, T, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # apply RoPE to q,k
        q = apply_rope(q, rope_sin, rope_cos)
        k = apply_rope(k, rope_sin, rope_cos)

        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        att = att.masked_fill(mask == 0, float('-inf'))
        att = torch.softmax(att, dim=-1)

        y = att @ v  # (B, heads, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.proj(y)


# SwiGLU FFN

In [7]:
class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim)
        self.w2 = nn.Linear(dim, hidden_dim)
        self.w3 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.w3(torch.nn.functional.silu(self.w1(x)) * self.w2(x))


# Transformer Block

In [8]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ffn_dim):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, num_heads)
        self.norm2 = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_dim)

    def forward(self, x, mask, rope_sin, rope_cos):
        x = x + self.attn(self.norm1(x), mask, rope_sin, rope_cos)
        x = x + self.ffn(self.norm2(x))
        return x


# Decoder Model

In [9]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, dim, num_layers, num_heads, ffn_dim, max_seq_len=2048):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.max_seq_len = max_seq_len

        self.token_emb = nn.Embedding(vocab_size, dim)

        self.blocks = nn.ModuleList([
            TransformerBlock(dim, num_heads, ffn_dim)
            for _ in range(num_layers)
        ])

        self.norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # weight tying
        self.lm_head.weight = self.token_emb.weight

        pos = torch.arange(max_seq_len)  # [T]
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2) / self.head_dim))
        # freqs len = head_dim/2

        sinusoid = torch.einsum("i,j->ij", pos, freqs)  # [T, head_dim/2]

        rope_sin = sinusoid.sin()   # [T, head_dim/2]
        rope_cos = sinusoid.cos()   # [T, head_dim/2]

        # Expand to: [1, 1, T, head_dim]
        rope_sin = torch.cat([rope_sin, rope_sin], dim=-1)
        rope_cos = torch.cat([rope_cos, rope_cos], dim=-1)

        self.register_buffer("rope_sin", rope_sin.unsqueeze(0).unsqueeze(0))
        self.register_buffer("rope_cos", rope_cos.unsqueeze(0).unsqueeze(0))
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, mask):
        B, T = idx.shape

        x = self.token_emb(idx)

        # Correct slice: [1,1,T,head_dim]
        rope_sin = self.rope_sin[:, :, :T, :]
        rope_cos = self.rope_cos[:, :, :T, :]

        for blk in self.blocks:
            x = blk(x, mask, rope_sin, rope_cos)

        x = self.norm(x)
        logits = self.lm_head(x)

        return logits


# Causal Mask

In [10]:
def causal_mask(T, device):
    m = torch.tril(torch.ones((T, T), dtype=torch.bool, device=device))
    return m.unsqueeze(0).unsqueeze(0)

# Dataset Class

In [11]:
class LMDataset(Dataset):
    def __init__(self, file, tokenizer, seq_len=2048):
        self.seq_len = seq_len
        self.tokenizer = tokenizer
        self.examples = []

        # We need seq_len + 1 tokens to create a shifted pair of (seq_len)
        chunk_size = seq_len + 1 

        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                ids = tokenizer.encode(line).ids
                
                # specific check to ensure we have enough tokens for x AND y
                if len(ids) < chunk_size:
                    continue
                
                # We stride by seq_len so we don't miss data, 
                # but we slice chunk_size (seq_len + 1) to get the target.
                for i in range(0, len(ids) - chunk_size + 1, seq_len):
                    chunk = ids[i : i + chunk_size]
                    if len(chunk) == chunk_size:
                        self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        # Load the chunk of size (seq_len + 1)
        data = torch.tensor(self.examples[idx], dtype=torch.long)
        
        # x is 0 to T-1
        x = data[:-1]
        
        # y is 1 to T (The prediction targets)
        y = data[1:]
        
        # Sanity check (can be removed later)
        assert len(x) == self.seq_len
        assert len(y) == self.seq_len
        
        return x, y

In [12]:
DATA_DIR = "../prepared"
TRAIN_FILE = os.path.join(DATA_DIR, "lm_train.txt")
VAL_FILE = os.path.join(DATA_DIR, "lm_val.txt")
TOKENIZER_FILE = "../tokens/tokenizer.json"
OUTPUT_DIR = "checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOCAB_SIZE = 16000
DIM = 256
NUM_LAYERS = 6
NUM_HEADS = 8
FFN_DIM = 1024
MAX_SEQ_LEN = 128

SEQ_LEN = 128
PHYSICAL_BATCH = 1
GRAD_ACCUM_STEPS = 32
EPOCHS = 3
LR = 3e-4
WEIGHT_DECAY = 0.01
SAVE_EVERY_STEPS = 2000
VAL_EVERY_STEPS = 2000
LOG_EVERY = 50
USE_AMP = True
NUM_WORKERS = 0
PIN_MEMORY = True


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [14]:
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Devices:", torch.cuda.device_count())
print(torch.version.cuda)

CUDA Available: True
CUDA Devices: 1
12.1


In [15]:
def save_checkpoint(step, model, optimizer, scaler=None):
    ckpt = {
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict()
    }
    if scaler is not None:
        ckpt["scaler_state_dict"] = scaler.state_dict()
    fname = os.path.join(OUTPUT_DIR, f"ckpt_step_{step}.pt")
    torch.save(ckpt, fname)
    print("Saved checkpoint:", fname)

In [16]:
print("Loading tokenizer...")
tok = Tokenizer.from_file(TOKENIZER_FILE)

train_ds = LMDataset(TRAIN_FILE, tok, seq_len=SEQ_LEN)
val_ds = LMDataset(VAL_FILE, tok, seq_len=SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=PHYSICAL_BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=PHYSICAL_BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)

print("Creating model...")
model = TransformerLM(VOCAB_SIZE, DIM, NUM_LAYERS, NUM_HEADS, FFN_DIM, max_seq_len=MAX_SEQ_LEN).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print("Param count (M):", sum(p.numel() for p in model.parameters())/1e6)

scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type=="cuda"))

Loading tokenizer...
Creating model...
Param count (M): 10.410752


In [17]:
@torch.no_grad()
def evaluate(model, val_loader, device, max_batches=None):
    model.eval()
    total_loss = 0.0
    n = 0

    for i, (x, y) in enumerate(val_loader):
        x, y = x.to(device), y.to(device)

        T = x.size(1)
        mask = causal_mask(T, device)
        logits = model(x, mask)

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        total_loss += loss.item()
        n += 1
        if max_batches and n >= max_batches:
            break

    model.train()
    return total_loss / max(1, n)

In [18]:
def load_checkpoint(ckpt_path, model, optimizer, scaler=None, device="cuda"):
    print(f"Loading checkpoint from {ckpt_path} ...")
    checkpoint = torch.load(ckpt_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scaler and "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    step = checkpoint.get("step", 0)
    print(f"Checkpoint loaded. Resuming from step {step}")
    return step


In [19]:
from tqdm import tqdm

def get_lr(step, warmup_steps=500, max_lr=LR, min_lr=1e-6, total_steps=200000):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + cosine_decay * (max_lr - min_lr)


resume_from = None

if resume_from:
    global_step = load_checkpoint(resume_from, model, optimizer, scaler, device=device)
else:
    global_step = 0

model.train()
print(f"Begin training from step {global_step}")

for epoch in range(EPOCHS):
    epoch_start = time.time()
    epoch_loss_sum = 0.0
    epoch_token_count = 0

    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}")

    for batch_idx, (x, y) in pbar:
        x, y = x.to(device), y.to(device)
        T = x.size(1)
        mask = causal_mask(T, device)

        lr_now = get_lr(global_step)
        optimizer.param_groups[0]["lr"] = lr_now

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
            logits = model(x, mask)
            loss_full = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        instant_loss = loss_full.item()
        epoch_loss_sum += instant_loss * x.numel()
        epoch_token_count += x.numel()

        loss = loss_full / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0).item()

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            global_step += 1
            avg_epoch_loss = epoch_loss_sum / epoch_token_count

            if global_step % LOG_EVERY == 0:
                pbar.set_postfix({
                    "inst_loss": f"{instant_loss:.6f}",
                    "avg_epoch": f"{avg_epoch_loss:.6f}",
                    "grad": f"{grad_norm:.6f}",
                    "lr": f"{lr_now:.6f}"
                })

            if global_step % VAL_EVERY_STEPS == 0:
                val_loss = evaluate(model, val_loader, device, max_batches=100)
                print(f"[VAL] step {global_step} | loss {val_loss:.6f}")
                save_checkpoint(global_step, model, optimizer, scaler)

            elif global_step % SAVE_EVERY_STEPS == 0:
                save_checkpoint(global_step, model, optimizer, scaler)

    epoch_loss_final = epoch_loss_sum / epoch_token_count
    print(f"Epoch {epoch+1} done | loss {epoch_loss_final:.6f} | time {time.time() - epoch_start:.2f}s")

save_checkpoint(global_step, model, optimizer, scaler)
print("Training complete.")


Begin training from step 0


Epoch 1:  55%|█████▍    | 64001/116689 [45:38<1:55:52,  7.58it/s, inst_loss=5.111506, avg_epoch=5.537039, grad=1.103585, lr=0.000300]

[VAL] step 2000 | loss 4.464011
Saved checkpoint: checkpoints\ckpt_step_2000.pt


Epoch 1: 100%|██████████| 116689/116689 [1:21:36<00:00, 23.83it/s, inst_loss=4.490269, avg_epoch=4.979494, grad=1.251042, lr=0.000300]


Epoch 1 done | loss 4.967069 | time 4896.98s


...

[VAL] step 8000 | loss 3.749357
Saved checkpoint: checkpoints\ckpt_step_8000.pt


Epoch 3:  74%|███████▍  | 86655/116689 [59:10<20:22, 24.57it/s, inst_loss=3.826791, avg_epoch=3.385758, grad=1.323436, lr=0.000298]  

[VAL] step 10000 | loss 3.683506
Saved checkpoint: checkpoints\ckpt_step_10000.pt


Epoch 3: 100%|██████████| 116689/116689 [1:19:40<00:00, 24.41it/s, inst_loss=3.686224, avg_epoch=3.373312, grad=1.413383, lr=0.000298]


Epoch 3 done | loss 3.373261 | time 4780.69s
Saved checkpoint: checkpoints\ckpt_step_10938.pt
Training complete.
